In [27]:
import polars as pl
import numpy as np
import pandas as pd
from pathlib import Path
import sys
sys.path.append("/kaggle/input/models/fedorfesenko/metrcs/other/default/1")
from metrics import evaluate_ranking

from catboost import CatBoostRanker, Pool
from sklearn.model_selection import GroupShuffleSplit

import warnings
warnings.filterwarnings("ignore")



In [2]:
DATA_DIR = Path("/kaggle/input/datasets/fedorfesenko/train-interactions")

TRAIN_PATH = DATA_DIR / "train_dataset.parquet"
VAL_PATH   = DATA_DIR / "val_dataset.parquet"
TEST_PATH  = DATA_DIR / "test_dataset.parquet"

for p in [TRAIN_PATH, VAL_PATH, TEST_PATH]:
    print(f"{p} exists: {p.exists()}")

/kaggle/input/datasets/fedorfesenko/train-interactions/train_dataset.parquet exists: True
/kaggle/input/datasets/fedorfesenko/train-interactions/val_dataset.parquet exists: True
/kaggle/input/datasets/fedorfesenko/train-interactions/test_dataset.parquet exists: True


In [4]:
df_train = pl.read_parquet(TRAIN_PATH)
df_val   = pl.read_parquet(VAL_PATH)
df_test  = pl.read_parquet(TEST_PATH)

print(f"df_train: {df_train.shape}")
print(f"df_val:   {df_val.shape}")
print(f"df_test:  {df_test.shape}")

df_train: (1810569, 74)
df_val:   (323736, 74)
df_test:  (292826, 72)


In [9]:
NUM_FEATURES = [
    "n_views_all", "n_clicks_all", "n_cart_updates_all", "n_purchases_all",
    "n_views_30d", "n_clicks_30d", "n_cart_updates_30d", "n_purchases_30d",
    "n_views_7d", "n_clicks_7d", "n_cart_updates_7d", "n_purchases_7d",
    "n_views_1d", "n_clicks_1d", "n_cart_updates_1d", "n_purchases_1d",
    "days_since_first_action", "days_since_last_action",
    "item_n_views_all", "item_n_clicks_all", "item_n_cart_updates_all", "item_n_purchases_all",
    "item_n_views_30d", "item_n_clicks_30d", "item_n_cart_updates_30d", "item_n_purchases_30d",
    "item_n_views_7d", "item_n_clicks_7d", "item_n_cart_updates_7d", "item_n_purchases_7d",
    "item_n_views_1d", "item_n_clicks_1d", "item_n_cart_updates_1d", "item_n_purchases_1d",
    "item_n_unique_users_all", "item_n_unique_users_30d", "item_n_unique_users_7d", "item_n_unique_users_1d",
    "item_days_since_first_view", "item_days_since_last_view",
    "ui_n_views_all", "ui_n_clicks_all", "ui_n_cart_updates_all", "ui_n_purchases_all",
    "ui_days_since_last_action", "ui_days_since_last_cart_update", "ui_days_since_last_purchase",
    "uc_n_views_all", "uc_n_clicks_all", "uc_n_cart_updates_all", "uc_n_purchases_all",
    "uc_n_views_30d", "uc_n_clicks_30d", "uc_n_cart_updates_30d", "uc_n_purchases_30d",
    "uc_n_views_7d", "uc_n_clicks_7d", "uc_n_cart_updates_7d", "uc_n_purchases_7d",
    "uc_days_since_last_purchase",
    "als_score", "v5_score", "v6_score",
    "hour_of_day", "day_of_week",
]
CAT_FEATURES = ["source_type", "city_name"]
FEATURES = NUM_FEATURES + CAT_FEATURES

print(f"NUM_FEATURES: {len(NUM_FEATURES)}")
print(f"CAT_FEATURES: {len(CAT_FEATURES)}")
print(f"FEATURES total: {len(FEATURES)}")


missing_train = [f for f in FEATURES if f not in df_train.columns]
missing_val = [f for f in FEATURES if f not in df_val.columns]
missing_test = [f for f in FEATURES if f not in df_test.columns]
print(f"\nmissing in train: {missing_train}")
print(f"missing in val:   {missing_val}")
print(f"missing in test:  {missing_test}")

NUM_FEATURES: 65
CAT_FEATURES: 2
FEATURES total: 67

missing in train: []
missing in val:   []
missing in test:  []


In [11]:
def prepare_pool(df, features, cat_features, target_col="target", group_col="request_id"):
    df = df.sort(group_col)
    
    X = df.select(features).to_pandas()
    
    for c in features:
        if c not in cat_features:
            X[c] = X[c].astype("float32")
    
    for c in cat_features:
        X[c] = X[c].fillna("unknown").astype(str)
    
    if target_col in df.columns:
        y = df[target_col].to_numpy().astype("float32")
    else:
        y = np.zeros(len(df), dtype="float32")
    
    group_id = df[group_col].to_numpy()
    
    return Pool(data=X, label=y, group_id=group_id, cat_features=cat_features), df

In [12]:
print("Preparing train pool...")
train_pool, df_train_sorted = prepare_pool(df_train, FEATURES, CAT_FEATURES)
print(f"train_pool: {train_pool.num_row()} rows")

print("Preparing val pool...")
val_pool, df_val_sorted = prepare_pool(df_val, FEATURES, CAT_FEATURES)
print(f"val_pool: {val_pool.num_row()} rows")

print("Preparing test pool...")
test_pool, df_test_sorted = prepare_pool(df_test, FEATURES, CAT_FEATURES)
print(f"test_pool: {test_pool.num_row()} rows")

Preparing train pool...
train_pool: 1810569 rows
Preparing val pool...
val_pool: 323736 rows
Preparing test pool...
test_pool: 292826 rows


In [17]:
model_v1 = CatBoostRanker(
    iterations=1000,
    learning_rate=0.05,
    depth=8,
    loss_function="YetiRank",
    eval_metric="NDCG:top=10",
    task_type="CPU",
    thread_count=-1,
    verbose=100,
    random_seed=42,
    allow_writing_files=False,
)

In [18]:
model_v1.fit(
    train_pool,
    eval_set=val_pool,
    early_stopping_rounds=50,
    use_best_model=True,
    verbose=100,
)

Groupwise loss function. OneHotMaxSize set to 10
0:	test: 0.8429216	best: 0.8429216 (0)	total: 3.29s	remaining: 54m 50s
100:	test: 0.8764128	best: 0.8766672 (95)	total: 5m 50s	remaining: 51m 55s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.8774949127
bestIteration = 127

Shrink model to first 128 iterations.


CatBoostRanker(allow_writing_files=False, depth=8, eval_metric='NDCG:top=10', iterations=1000, learning_rate=0.05, loss_function='YetiRank', random_seed=42, task_type='CPU', verbose=100)

In [25]:
importance = model_v1.get_feature_importance(data=train_pool, type="FeatureImportance")
feat_names = model_v1.feature_names_

imp_df = pl.DataFrame({
    "feature": feat_names,
    "importance": importance,
}).sort("importance", descending=True)

print("=== Top-30 feature importance ===")
print(imp_df.head(30))
print(f"\n=== Bottom-10 ===")
print(imp_df.tail(10))

=== Top-30 feature importance ===
shape: (30, 2)
┌─────────────────────────┬────────────┐
│ feature                 ┆ importance │
│ ---                     ┆ ---        │
│ str                     ┆ f64        │
╞═════════════════════════╪════════════╡
│ als_score               ┆ 0.12595    │
│ v6_score                ┆ 0.110899   │
│ source_type             ┆ 0.006023   │
│ ui_n_cart_updates_all   ┆ 0.005489   │
│ ui_n_views_all          ┆ 0.005141   │
│ …                       ┆ …          │
│ item_n_cart_updates_all ┆ 0.000121   │
│ ui_n_purchases_all      ┆ 0.000105   │
│ uc_n_views_30d          ┆ 0.000098   │
│ item_n_unique_users_all ┆ 0.000097   │
│ uc_n_purchases_all      ┆ 0.000094   │
└─────────────────────────┴────────────┘

=== Bottom-10 ===
shape: (10, 2)
┌───────────────────────────┬────────────┐
│ feature                   ┆ importance │
│ ---                       ┆ ---        │
│ str                       ┆ f64        │
╞═══════════════════════════╪════════════╡
│ ite

In [28]:
val_preds = model_v1.predict(val_pool)

df_val_pred = pl.DataFrame({
    "request_id": df_val_sorted["request_id"].to_numpy(),
    "product_id": df_val_sorted["product_id"].to_numpy(),
    "target": df_val_sorted["target"].to_numpy(),
    "score": val_preds.astype("float64"),
})
print(f"df_val_pred: {df_val_pred.shape}")
print(df_val_pred.head(5))


df_val_pred_sorted = df_val_pred.sort(["request_id", "score"], descending=[False, True])

predicted_items = {}
true_items = {}
relevances = {}

for req_id, group in df_val_pred_sorted.group_by("request_id", maintain_order=True):
    req_id = req_id[0]
    items = group["product_id"].to_list()
    rels = group["target"].to_list()
    predicted_items[req_id] = items
    relevances[req_id] = rels
    # true_items: только те, у кого target=1
    true_items[req_id] = [item for item, rel in zip(items, rels) if rel == 1]

print(f"\nunique request_id: {len(predicted_items)}")
print(f"Пример для первого request_id:")
first_req = list(predicted_items.keys())[0]
print(f"  request_id={first_req}")
print(f"  predicted_items: {predicted_items[first_req][:10]}...")
print(f"  relevances: {relevances[first_req][:10]}...")
print(f"  true_items: {true_items[first_req]}")

df_val_pred: (323736, 4)
shape: (5, 4)
┌──────────────────┬──────────────────────┬────────┬──────────┐
│ request_id       ┆ product_id           ┆ target ┆ score    │
│ ---              ┆ ---                  ┆ ---    ┆ ---      │
│ u64              ┆ u64                  ┆ i8     ┆ f64      │
╞══════════════════╪══════════════════════╪════════╪══════════╡
│ 2146487132829478 ┆ 1892423684467877312  ┆ 0      ┆ 3.882454 │
│ 2146487132829478 ┆ 5435144058349499642  ┆ 1      ┆ 4.40591  │
│ 2146487132829478 ┆ 5435144058349499642  ┆ 1      ┆ 4.40591  │
│ 2146487132829478 ┆ 18119221307270730220 ┆ 0      ┆ 3.373224 │
│ 2146487132829478 ┆ 2107117453430345717  ┆ 0      ┆ 3.961255 │
└──────────────────┴──────────────────────┴────────┴──────────┘

unique request_id: 14149
Пример для первого request_id:
  request_id=2146487132829478
  predicted_items: [5435144058349499642, 5435144058349499642, 2107117453430345717, 1892423684467877312, 18119221307270730220, 14589993416551700732, 6730626684147470052, 9

In [29]:
metrics_v1 = evaluate_ranking(
    predicted_items=predicted_items,
    true_items=true_items,
    relevances=relevances,
    k=10,
)


In [30]:
metrics_v1

{'ndcg@10': 0.5584645280200674,
 'recall@10': 0.338238411898068,
 'mrr@10': 0.4492272858740088,
 'hit_rate@10': 0.7617623062279032,
 'map@10': 1.0639790393841637,
 'precision@10': 0.3398576954124955,
 'n_requests': 3677}

In [31]:
test_preds = model_v1.predict(test_pool)
print(f"test_preds shape: {test_preds.shape}")
print(f"min: {test_preds.min():.4f}, max: {test_preds.max():.4f}, mean: {test_preds.mean():.4f}")

df_submission = pl.DataFrame({
    "index": df_test_sorted["index"].to_numpy(),
    "request_id": df_test_sorted["request_id"].to_numpy(),
    "score": test_preds.astype("float64"),
})
print(f"\ndf_submission: {df_submission.shape}")
print(df_submission.head(10))

test_preds shape: (292826,)
min: -3.3626, max: 5.1953, mean: -1.1606

df_submission: (292826, 3)
shape: (10, 3)
┌────────┬──────────────────┬───────────┐
│ index  ┆ request_id       ┆ score     │
│ ---    ┆ ---              ┆ ---       │
│ u32    ┆ u64              ┆ f64       │
╞════════╪══════════════════╪═══════════╡
│ 215380 ┆ 9122459158330783 ┆ -2.42062  │
│ 183407 ┆ 9122459158330783 ┆ 2.955265  │
│ 205453 ┆ 9122459158330783 ┆ -2.500478 │
│ 26687  ┆ 9122459158330783 ┆ -2.35635  │
│ 8750   ┆ 9122459158330783 ┆ -2.254161 │
│ 8750   ┆ 9122459158330783 ┆ -2.254161 │
│ 222780 ┆ 9122459158330783 ┆ -2.303931 │
│ 27025  ┆ 9122459158330783 ┆ -2.084694 │
│ 86493  ┆ 9122459158330783 ┆ -2.089733 │
│ 181186 ┆ 9122459158330783 ┆ -2.084694 │
└────────┴──────────────────┴───────────┘


In [32]:
df_submission_sorted = df_submission.sort(
    ["request_id", "score"],
    descending=[False, True],
)

df_final = df_submission_sorted.select(["index", "request_id"])


df_final.write_csv("submission.csv")
print(f"submission.csv saved: {df_final.shape}")
print(df_final.head(10))

submission.csv saved: (292826, 2)
shape: (10, 2)
┌────────┬──────────────────┐
│ index  ┆ request_id       │
│ ---    ┆ ---              │
│ u32    ┆ u64              │
╞════════╪══════════════════╡
│ 183407 ┆ 9122459158330783 │
│ 27025  ┆ 9122459158330783 │
│ 181186 ┆ 9122459158330783 │
│ 86493  ┆ 9122459158330783 │
│ 8750   ┆ 9122459158330783 │
│ 8750   ┆ 9122459158330783 │
│ 222780 ┆ 9122459158330783 │
│ 26687  ┆ 9122459158330783 │
│ 215380 ┆ 9122459158330783 │
│ 205453 ┆ 9122459158330783 │
└────────┴──────────────────┘


In [47]:
model_v3 = CatBoostRanker(
    iterations=1000,
    learning_rate=0.05,
    depth=8,
    loss_function="YetiRank",
    eval_metric="NDCG:top=10",
    task_type="CPU",
    thread_count=-1,
    verbose=100,
    random_seed=42,
    allow_writing_files=False,
)

In [48]:
model_v3.fit(
    train_pool_v3,
    eval_set=val_pool_v3,
    early_stopping_rounds=50,
    use_best_model=True,
    verbose=100,
)

Groupwise loss function. OneHotMaxSize set to 10
0:	test: 0.8439602	best: 0.8439602 (0)	total: 3.76s	remaining: 1h 2m 41s
100:	test: 0.8794375	best: 0.8799822 (80)	total: 6m 59s	remaining: 1h 2m 16s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.8799821583
bestIteration = 80

Shrink model to first 81 iterations.


CatBoostRanker(allow_writing_files=False, depth=8, eval_metric='NDCG:top=10', iterations=1000, learning_rate=0.05, loss_function='YetiRank', random_seed=42, task_type='CPU', verbose=100)

In [49]:
test_preds_v3 = model_v3.predict(test_pool_v3)
print(f"test_preds_v3: {test_preds_v3.shape}")
print(f"min: {test_preds_v3.min():.4f}, max: {test_preds_v3.max():.4f}, mean: {test_preds_v3.mean():.4f}")

df_sub_v3 = pl.DataFrame({
    "index": df_test_sorted_v3["index"].to_numpy(),
    "request_id": df_test_sorted_v3["request_id"].to_numpy(),
    "score": test_preds_v3.astype("float64"),
})

df_sub_v3_sorted = df_sub_v3.sort(["request_id", "score"], descending=[False, True])
df_sub_v3_final = df_sub_v3_sorted.select(["index", "request_id"])
df_sub_v3_final.write_csv("submission_v3.csv")

print(f"\nsubmission_v3.csv: {df_sub_v3_final.shape}")
print(f"unique index: {df_sub_v3_final['index'].n_unique()}")
print(df_sub_v3_final.head(5))

test_preds_v3: (292826,)
min: -2.6460, max: 4.3061, mean: -1.0098

submission_v3.csv: (292826, 2)
unique index: 232166
shape: (5, 2)
┌────────┬──────────────────┐
│ index  ┆ request_id       │
│ ---    ┆ ---              │
│ u32    ┆ u64              │
╞════════╪══════════════════╡
│ 183407 ┆ 9122459158330783 │
│ 215380 ┆ 9122459158330783 │
│ 86493  ┆ 9122459158330783 │
│ 8750   ┆ 9122459158330783 │
│ 8750   ┆ 9122459158330783 │
└────────┴──────────────────┘


In [51]:
sub_v3 = pl.read_csv(
    "submission_v3.csv",
    schema_overrides={"request_id": pl.UInt64, "index": pl.UInt64},
)
print(f"sub_v3 shape: {sub_v3.shape}")
print(f"columns: {sub_v3.columns}")
print(f"unique index: {sub_v3['index'].n_unique()}")
print(sub_v3.head(5))

df_test_orig = pl.read_parquet(DATA_DIR / "df_test.parquet")
test_indices = set(df_test_orig["index"].to_list())
sub_indices = set(sub_v3["index"].to_list())
print(f"\ntest_indices - sub_indices: {len(test_indices - sub_indices)}")
print(f"sub_indices - test_indices: {len(sub_indices - test_indices)}")

sub_v3 shape: (292826, 2)
columns: ['index', 'request_id']
unique index: 232166
shape: (5, 2)
┌────────┬──────────────────┐
│ index  ┆ request_id       │
│ ---    ┆ ---              │
│ u64    ┆ u64              │
╞════════╪══════════════════╡
│ 183407 ┆ 9122459158330783 │
│ 215380 ┆ 9122459158330783 │
│ 86493  ┆ 9122459158330783 │
│ 8750   ┆ 9122459158330783 │
│ 8750   ┆ 9122459158330783 │
└────────┴──────────────────┘

test_indices - sub_indices: 60660
sub_indices - test_indices: 0


In [52]:
key_cols = ["user_id", "product_id", "request_id"]
feature_cols_v3 = [c for c in df_test_v3.columns if c not in ["index", "timestamp", "source_type", "city_name", "product_category", "hour_of_day", "day_of_week"] + key_cols]

cols_to_keep = [c for c in df_test_v3.columns if c != "index"]

df_test_v3_features = (
    df_test_v3
    .select(cols_to_keep)
    .unique(subset=key_cols, keep="first")
)
print(f"df_test_v3_features: {df_test_v3_features.shape}")

df_test_v3_rebuilt = df_test_orig.join(
    df_test_v3_features,
    on=key_cols,
    how="left",
)
print(f"df_test_v3_rebuilt: {df_test_v3_rebuilt.shape}")
print(f"unique index: {df_test_v3_rebuilt['index'].n_unique()}")
print(f"null count: {df_test_v3_rebuilt.null_count().sum_horizontal()}")

drop_right = [c for c in df_test_v3_rebuilt.columns if c.endswith("_right")]
if drop_right:
    print(f"Drop right cols: {drop_right}")
    df_test_v3_rebuilt = df_test_v3_rebuilt.drop(drop_right)

print(f"\ncolumns: {df_test_v3_rebuilt.columns}")
print(f"shape: {df_test_v3_rebuilt.shape}")

df_test_v3_features: (232166, 72)
df_test_v3_rebuilt: (292826, 80)
unique index: 292826
null count: shape: (1,)
Series: 'sum' [u32]
[
	1640145
]
Drop right cols: ['timestamp_right', 'source_type_right', 'city_name_right', 'product_category_right']

columns: ['index', 'city_name', 'product_category', 'product_id', 'product_image', 'product_name', 'request_id', 'source_type', 'store_id', 'user_id', 'timestamp', 'n_views_all', 'n_clicks_all', 'n_cart_updates_all', 'n_purchases_all', 'n_views_30d', 'n_clicks_30d', 'n_cart_updates_30d', 'n_purchases_30d', 'n_views_7d', 'n_clicks_7d', 'n_cart_updates_7d', 'n_purchases_7d', 'n_views_1d', 'n_clicks_1d', 'n_cart_updates_1d', 'n_purchases_1d', 'days_since_first_action', 'days_since_last_action', 'item_n_views_all', 'item_n_clicks_all', 'item_n_cart_updates_all', 'item_n_purchases_all', 'item_n_views_30d', 'item_n_clicks_30d', 'item_n_cart_updates_30d', 'item_n_purchases_30d', 'item_n_views_7d', 'item_n_clicks_7d', 'item_n_cart_updates_7d', 'item

In [53]:
test_pool_v3_fixed, df_test_sorted_v3_fixed = prepare_pool(
    df_test_v3_rebuilt, FEATURES_V3, CAT_FEATURES_V3
)
print(f"test_pool_v3_fixed: {test_pool_v3_fixed.num_row()}")

test_preds_v3_fixed = model_v3.predict(test_pool_v3_fixed)
print(f"test_preds_v3_fixed: {test_preds_v3_fixed.shape}")

df_sub_v3_fixed = pl.DataFrame({
    "index": df_test_sorted_v3_fixed["index"].to_numpy(),
    "request_id": df_test_sorted_v3_fixed["request_id"].to_numpy(),
    "score": test_preds_v3_fixed.astype("float64"),
})
df_sub_v3_fixed_sorted = df_sub_v3_fixed.sort(["request_id", "score"], descending=[False, True])
df_sub_v3_fixed_final = df_sub_v3_fixed_sorted.select(["index", "request_id"])
df_sub_v3_fixed_final.write_csv("submission_v3_fixed.csv")

print(f"\nsubmission_v3_fixed.csv: {df_sub_v3_fixed_final.shape}")
print(f"unique index: {df_sub_v3_fixed_final['index'].n_unique()}")
print(df_sub_v3_fixed_final.head(5))

test_pool_v3_fixed: 292826
test_preds_v3_fixed: (292826,)

submission_v3_fixed.csv: (292826, 2)
unique index: 292826
shape: (5, 2)
┌────────┬──────────────────┐
│ index  ┆ request_id       │
│ ---    ┆ ---              │
│ u32    ┆ u64              │
╞════════╪══════════════════╡
│ 183407 ┆ 9122459158330783 │
│ 215380 ┆ 9122459158330783 │
│ 86493  ┆ 9122459158330783 │
│ 230321 ┆ 9122459158330783 │
│ 8750   ┆ 9122459158330783 │
└────────┴──────────────────┘


In [54]:
imp_v3 = model_v3.get_feature_importance(data=train_pool_v3, type="FeatureImportance")
imp_df_v3 = pl.DataFrame({
    "feature": model_v3.feature_names_,
    "importance": imp_v3,
}).sort("importance", descending=True)

print(imp_df_v3.head(20))
pc = imp_df_v3.with_row_index("rank").filter(pl.col("feature") == "product_category")
print(pc)

shape: (20, 2)
┌────────────────────────────┬────────────┐
│ feature                    ┆ importance │
│ ---                        ┆ ---        │
│ str                        ┆ f64        │
╞════════════════════════════╪════════════╡
│ v6_score                   ┆ 0.077449   │
│ als_score                  ┆ 0.073512   │
│ source_type                ┆ 0.016519   │
│ ui_n_cart_updates_all      ┆ 0.00746    │
│ ui_n_views_all             ┆ 0.004274   │
│ …                          ┆ …          │
│ n_cart_updates_all         ┆ 0.000292   │
│ item_n_views_30d           ┆ 0.000234   │
│ uc_n_views_all             ┆ 0.000191   │
│ item_days_since_first_view ┆ 0.000136   │
│ uc_n_clicks_all            ┆ 0.000133   │
└────────────────────────────┴────────────┘
shape: (1, 3)
┌──────┬──────────────────┬────────────┐
│ rank ┆ feature          ┆ importance │
│ ---  ┆ ---              ┆ ---        │
│ u32  ┆ str              ┆ f64        │
╞══════╪══════════════════╪════════════╡
│ 7    ┆ product_c